Load data

In [5]:
import pandas as pd
import numpy as np

standings = pd.read_csv("data/raw/team_stats_2003_2025.csv")
playoffs = pd.read_csv("data/raw/nfl_playoffs_2003_2025.csv")
markets = pd.read_csv("data/raw/nfl_markets.csv")
payroll = pd.read_csv("data/raw/nfl_payroll_2011_2025.csv")
draft = pd.read_csv("data/raw/nfl_first_round_picks_2003_2025.csv")
revenue = pd.read_csv("data/raw/nfl_national_revenue_2011_2025.csv")
champions = pd.read_csv("data/raw/super_bowl_champions_2003_2025.csv")

assert standings.groupby("year").size().eq(32).all()
print("standings:", standings.shape)



standings: (736, 35)


Keep relevant data in standings data

In [6]:
trimTO = ["year","team", "wins","losses","ties","win_loss_perc", "g"]
df = standings[trimTO].copy()

#verify win% is calculated correctly
exact = (df["wins"] + 0.5 * df["ties"]) / df["g"]
assert (exact - df["win_loss_perc"]).abs().max() < 0.0006, "win% does not match (W + T/2) / G"
assert (df["wins"] + df["losses"] + df["ties"] == df["g"]).all()
df["win_loss_perc"] = exact

Take into account Franchise relocation to have consistend FranchiseIDs

In [7]:
franchise = {"Oakland Raiders": "LV", "Las Vegas Raiders": "LV",
             "San Diego Chargers": "LAC" , "Los Angeles Chargers": "LAC",
             "St. Louis Rams": "LAR", "Los Angeles Rams":"LAR",
             "Washington Redskins": "WAS", "Washington Football Team": "WAS", "Washington Commanders":"WAS",
             "Miami Dolphins": "MIA",
             "Carolina Panthers": "CAR",
             "Atlanta Falcons":"ATL",
             "Minnesota Vikings":"MIN",
             "Chicago Bears":"CHI",
             "Tennessee Titans":"TEN",
             "Cincinnati Bengals":"CIN",
             "Buffalo Bills":"BUF",
             "New England Patriots":"NE",
             "New York Jets":"NYJ",
             "Dallas Cowboys":"DAL",
             "New York Giants":"NYG",
             "Philadelphia Eagles":"PHI",
             "Baltimore Ravens":"BAL",
             "Cleveland Browns":"CLV",
             "Pittsburgh Steelers":"PIT",
             "Detroit Lions":"DET",
             "Green Bay Packers":"GB",
             "Houston Texans":"HST",
             "Indianapolis Colts":"IND",
             "Jacksonville Jaguars":"JAX",
             "New Orleans Saints":"NO",
             "Tampa Bay Buccaneers":"TB",
             "Denver Broncos":"DEN",
             "Kansas City Chiefs":"KC",
             "Arizona Cardinals":"ARZ",
             "San Francisco 49ers":"SF",
             "Seattle Seahawks":"SEA",
             }
df["franchise"] = df["team"].map(franchise)
assert df["franchise"].notna().all(), df.loc[df["franchise"].isna(), "team"].unique()
assert df.groupby("year")["franchise"].nunique().eq(32).all()

Join Playoff Data

In [8]:
df = df.merge(playoffs, on=["year", "team"], how="left", validate="one_to_one")
assert df["made_playoffs"].notna().all()

#Verify Data Integrity
expected = np.where(df.groupby("year").size().index >= 2020, 14, 12)
assert (df.groupby("year")["made_playoffs"].sum().values == expected).all()

sb = df.loc[df["won_sb"] == 1, ["year", "team"]].set_index("year")["team"]
assert sb.index.is_unique and len(sb) == 23
assert (sb.reindex(champions["season"]).values == champions["champion"].values).all()

Join markets size to the data

In [9]:
mk = markets.copy()
mk["franchise"] = mk["team"].map(franchise)
assert mk["franchise"].notna().all()

mk = mk[["franchise", "valid_from", "valid_to", "dma_name", "dma_rank","teams_in_market", "census_metro_name", "metro_pop_2025", "pop_per_team_2025"]]

df = df.merge(mk, on="franchise", how="left")
df = df[df["year"].between(df["valid_from"], df["valid_to"])].drop(columns=["valid_from", "valid_to"])
assert len(df) == 736, f"Expected 736 rows, got {len(df)}"
assert df.groupby("year").size().eq(32).all()
assert df["pop_per_team_2025"].notna().all()

Join Payroll and Revenue share per team, rows before 2011 stay blank and will not be used for regression as payroll data before 2011 difficult to find

In [10]:
pay = payroll[["year","team", "total_cap", "league_cap", "rel_payroll"]]
df = df.merge(pay, on=["year", "team"], how="left", validate="one_to_one")
assert df.loc[df["year"] >= 2011, "total_cap"].notna().all()

rev = revenue[["year", "national_share_per_team_musd", "league_national_revenue_busd"]]
df = df.merge(rev, on="year", how="left", validate="many_to_one")



Add to indicate Prior Season Win%, Log population for regression usage as Large cities will over inflate, and tiers for markets

In [11]:
df = df.sort_values(["franchise", "year"]).reset_index(drop=True)
df["prior_win_pct"] = df.groupby("franchise")["win_loss_perc"].shift(1)
df["log_pop"] = np.log(df["pop_per_team_2025"])

current = df.loc[df["year"] == 2025, "pop_per_team_2025"]
cuts = [0, current.quantile(1 / 3), current.quantile(2 / 3), np.inf]
df["market_tier"] = pd.cut(df["pop_per_team_2025"], bins=cuts, labels=["Low", "Mid", "High"])
print("Tier cut points (people per team):", [round(c) for c in cuts[1:3]])
print(df[df["year"] == 2025]["market_tier"].value_counts())

Tier cut points (people per team): [2567255, 5962391]
market_tier
Low     11
High    11
Mid     10
Name: count, dtype: int64


Assign Values using jimmy johnson table to picks and add draft data

In [12]:
draft["franchise"] = draft["team"].map(franchise)
assert draft["franchise"].notna().all(), draft.loc[draft["franchise"].isna(), "team"].unique()

jimmy_johnson_points = {
    1: 3000,  2: 2600,  3: 2200,  4: 1800,  5: 1700,  6: 1600,  7: 1500,  8: 1400,
    9: 1350, 10: 1300, 11: 1250, 12: 1200, 13: 1150, 14: 1100, 15: 1050, 16: 1000,
    17:  950, 18:  900, 19:  875, 20:  850, 21:  800, 22:  780, 23:  760, 24:  740,
    25:  720, 26:  700, 27:  680, 28:  660, 29:  640, 30:  620, 31:  600, 32:  590,
}
draft["pick_value"] = draft["pick"].map(jimmy_johnson_points)
assert draft["pick_value"].notna().all()

dv = (draft.groupby(["franchise", "season"])
      .agg(pick_value_sum=("pick_value", "sum"), first_round_picks=("pick", "size"), best_pick=("pick", "min"))
      .reset_index()
      .rename(columns={"season": "year"}))

grid = pd.MultiIndex.from_product([sorted(df["franchise"].unique()), range(2003, 2026)], names=["franchise", "year"])
dv = dv.set_index(["franchise", "year"]).reindex(grid).reset_index()
dv[["pick_value_sum", "first_round_picks"]] = dv[["pick_value_sum", "first_round_picks"]].fillna(0)

dv = dv.sort_values(["franchise", "year"])
dv["draft_capital_3yr"] = dv.groupby("franchise")["pick_value_sum"].transform(lambda x: x.shift(1).rolling(3, min_periods=3).sum())

df = df.merge(dv[["franchise", "year", "draft_capital_3yr", "best_pick"]], on=["franchise", "year"], how="left", validate="one_to_one")
df = df.rename(columns={"best_pick": "best_pick_this_year"})

assert df.loc[df["year"] >= 2006, "draft_capital_3yr"].notna().all()

Seasonal Outcomes for Glyph pie charts

In [13]:
outcome_map = {
    "Missed playoffs": "Missed playoffs",
    "Wild Card": "Lost early in playoffs",
    "Divisional": "Lost early in playoffs",
    "Conference Championship": "Lost in conference title game",
    "Super Bowl": "Lost in Super Bowl",
    "Won Super Bowl": "Won Super Bowl",
}
OUTCOMES = ["Missed playoffs", "Lost early in playoffs", "Lost in conference title game",
            "Lost in Super Bowl", "Won Super Bowl"]
df["season_outcome"] = df["furthest_round"].map(outcome_map)
assert df["season_outcome"].notna().all()

Final Checks for Main Table

In [14]:
assert len(df) == 736, f"Expected 736 rows, got {len(df)}"
assert df.groupby("year").size().eq(32).all()
assert not df.duplicated(subset=["year", "franchise"]).any(), "Duplicate franchise-year combinations found"
assert df.groupby("year")["won_sb"].sum().eq(1).all()

cols = ["year", "team", "franchise", "wins", "losses", "ties", "g", "win_loss_perc",
        "made_playoffs", "had_bye", "playoff_games", "playoff_wins", "furthest_round",
        "season_outcome", "won_sb",
        "dma_name", "dma_rank", "teams_in_market", "census_metro_name", "metro_pop_2025",
        "pop_per_team_2025", "log_pop", "market_tier",
        "total_cap", "league_cap", "rel_payroll",
        "national_share_per_team_musd", "league_national_revenue_busd",
        "prior_win_pct", "draft_capital_3yr", "best_pick_this_year"]
df = df[cols].sort_values(["year", "team"]).reset_index(drop=True)

missing = df.isna().sum()
print(missing[missing > 0])
df.to_csv("data/processed/team_seasons.csv", index=False)
print("Saved teams Seasons csv:", df.shape)

total_cap                       256
league_cap                      256
rel_payroll                     256
national_share_per_team_musd    256
league_national_revenue_busd    256
prior_win_pct                    32
draft_capital_3yr                96
best_pick_this_year              96
dtype: int64
Saved teams Seasons csv: (736, 31)


Data for Regression

In [15]:
regression = df[(df["year"] >= 2011)][["year","team","franchise","win_loss_perc","made_playoffs","market_tier","dma_rank", "log_pop", "pop_per_team_2025", "draft_capital_3yr", "prior_win_pct","rel_payroll"]].dropna().copy()
assert len(regression) == 480, f"expected 480 rows, got {len(regression)}"
regression.to_csv("data/processed/regression_data.csv", index=False)
print("Regression data:", regression.shape)

Regression data: (480, 12)


Market Size Summary

In [16]:
df = pd.read_csv("data/processed/team_seasons.csv")

tier_2025 = df.loc[df["year"] == 2025].set_index("franchise")["market_tier"]
df["franchise_tier"] = df["franchise"].map(tier_2025)

franchise_level = (df.groupby(["franchise", "franchise_tier"])
                     .agg(win_pct=("win_loss_perc", "mean"),
                          playoff_seasons=("made_playoffs", "sum"),
                          playoff_wins=("playoff_wins", "sum"),
                          titles=("won_sb", "sum"))
                     .reset_index())

tier_summary = (franchise_level.groupby("franchise_tier")
                  .agg(franchises=("franchise", "size"),
                       avg_win_pct=("win_pct", "mean"),
                       avg_playoff_seasons=("playoff_seasons", "mean"),
                       avg_playoff_wins=("playoff_wins", "mean"),
                       total_titles=("titles", "sum"))
                  .reindex(["Low", "Mid", "High"])
                  .round(3))
tier_summary

,franchises,avg_win_pct,avg_playoff_seasons,avg_playoff_wins,total_titles
franchise_tier,,,,,
Low,11,0.501,9.182,8.091,8
Mid,10,0.521,10.000,10.700,10
High,11,0.480,7.909,6.273,5
